In [1]:
# CELL 0: MANUAL AUTHENTICATION
import os

from kaggle_secrets import UserSecretsClient

try:
    user_secrets = UserSecretsClient()
    os.environ["KAGGLE_USERNAME"] = user_secrets.get_secret("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = user_secrets.get_secret("KAGGLE_KEY")
    print("Authentication variables set from Secrets.")
except Exception as e:
    print(f"Error: Could not retrieve secrets. {e}")

# This part creates the physical file the SDK sometimes looks for
!mkdir -p ~/.kaggle
!echo '{{"username":"{os.environ["KAGGLE_USERNAME"]}","key":"{os.environ["KAGGLE_KEY"]}"}}' > ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json

print("Kaggle API Credentials successfully activated and file-injected.")

Authentication variables set from Secrets.
Kaggle API Credentials successfully activated and file-injected.


In [3]:
# CELL 1: SETUP
!pip install -q 'datasets>=2.20,<4.0' 'scipy>=1.14,<2.0' 'numpy>=2.0,<3.0' 'pandas>=2.0,<3.0'
!pip install -q kaggle-benchmarks  # 2026 Evaluation SDK
!pip install 'google-cloud-bigquery-storage>=2.0.0,<3.0.0'

import json
import os
import random

import numpy as np
from scipy.stats import pearsonr

# Define working directory for Kaggle
_IS_KAGGLE = os.path.isdir('/kaggle')
WORKING_DIR = "/kaggle/working/kaggle_payloads" if _IS_KAGGLE else "./kaggle_payloads"
RESULTS_DIR = "/kaggle/working/kaggle_results" if _IS_KAGGLE else "./kaggle_results"

os.makedirs(WORKING_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print("Environment ready. Directories created.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.4/304.4 kB 13.0 MB/s eta 0:00:00
Environment ready. Directories created.


In [4]:
# CELL 2: PAYLOAD GENERATION — N=200 HUMAN BASELINE
import os

# --- SEPARATE READ AND WRITE PATHS FOR KAGGLE ---
SCRIPT_DIR = "/kaggle/input/datasets/[ANONYMOUS]/sigma-base-datasets" if _IS_KAGGLE else "../datasets"
OUTPUT_DIR  = WORKING_DIR
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ──────────────────────────────────────────────────────────────────────────────
# DATA LOADERS  (raise on missing — no silent fallback)
# ──────────────────────────────────────────────────────────────────────────────

def load_local_cogs(split):
    split_name = "dev" if split == "validation" else split
    path = os.path.join(SCRIPT_DIR, "COGS", "COGS", "data", f"{split_name}.tsv")
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing COGS file: {path}")
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split("\t")
            if len(parts) >= 2:
                data.append({"source": parts[0], "target": parts[1]})
    return data

def load_local_scan(split):
    path = os.path.join(SCRIPT_DIR, "SCAN", "SCAN", "simple_split", f"tasks_{split}_simple.txt")
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing SCAN file: {path}")
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if " OUT: " in line:
                src, tgt = line.strip().split(" OUT: ")
                src = src.replace("IN: ", "").strip()
                data.append({"commands": src, "actions": tgt.strip()})
    return data

def load_local_pcfg(split):
    base_path = os.path.join(SCRIPT_DIR, "am-i-compositional", "am-i-compositional",
                              "data", "pcfgset", "systematicity")
    src_path = f"{base_path}/{split}.src"
    tgt_path = f"{base_path}/{split}.tgt"
    if not os.path.exists(src_path) or not os.path.exists(tgt_path):
        raise FileNotFoundError(f"Missing PCFG files at {base_path}")
    data = []
    with open(src_path, "r", encoding="utf-8") as fs, \
         open(tgt_path, "r", encoding="utf-8") as ft:
        for s_line, t_line in zip(fs, ft):
            s, t = s_line.strip(), t_line.strip()
            if s and t:
                data.append({"source": s, "target": t})
    return data

# ──────────────────────────────────────────────────────────────────────────────
# SHARED UTILITIES
# ──────────────────────────────────────────────────────────────────────────────

def format_for_kaggle(
    task_id: str | int,
    prompt: str,
    expected: str,
    track_name: str,
    condition: str,
    temperature: float = 0.0,
) -> dict:
    """Returns one item in the Kaggle Community Benchmarks SDK schema."""
    return {
        "id": f"{track_name}_{condition}_{task_id}",
        "prompt": prompt,
        "expected": expected,
        "temperature": temperature,
    }

def write_jsonl(path: str, records: list[dict]) -> None:
    filename = os.path.basename(path)
    safe_path = os.path.join(OUTPUT_DIR, filename)
    with open(safe_path, "w") as f:
        for r in records:
            f.write(json.dumps(r) + "\n")
    print(f"  Wrote {len(records)} items -> {safe_path}")

# ──────────────────────────────────────────────────────────────────────────────
# TRACK 1 — LEARNING (H-PTB)  N=200 per condition
# Conditions: A (Random/IID), B (Difficulty-Ordered),
#             C (Structure-Targeted), D (Structure-Targeted + αA-Building)
# ──────────────────────────────────────────────────────────────────────────────

def generate_hptb() -> None:
    print("\n[1/5] Generating H-PTB (Learning) — N=200 per condition ...")
    train_data = load_local_pcfg("train")          # raises if absent

    payload = []
    train_copy = list(train_data)
    random.Random(42).shuffle(train_copy)

    # Condition A — Baseline (Random/IID): 200 items
    iid = train_copy[:200]
    for i, item in enumerate(iid):
        payload.append(format_for_kaggle(
            f"A_{i}",
            f"Translate the following sequence into its action representation:\n{item['source']}",
            item["target"], "H-PTB", "Cond_A"))

    # Condition B — Difficulty-Ordered: 200 items (non-overlapping slice)
    sorted_by_difficulty = sorted(train_copy[200:600], key=lambda x: len(x["source"].split()))
    for i, item in enumerate(sorted_by_difficulty[:200]):
        payload.append(format_for_kaggle(
            f"B_{i}",
            f"[Curriculum: easy-first] Translate the following sequence into its action representation:\n{item['source']}",
            item["target"], "H-PTB", "Cond_B"))

    # Conditions C & D — Contrastive pairs: up to 200 pairs each
    groups = {}
    for item in train_data[:10000]:
        groups.setdefault(frozenset(item["source"].split()), []).append(
            (item["source"], item["target"]))

    contrastive_idx = 0
    for items in groups.values():
        unique = list({tgt: (src, tgt) for src, tgt in items}.values())
        if len(unique) < 2:
            continue
        (src_a, tgt_a), (_, tgt_b) = unique[0], unique[1]
        if tgt_a == tgt_b:
            continue

        payload.append(format_for_kaggle(
            f"C_{contrastive_idx}",
            f"Which output strictly follows the compositional rule?\n"
            f"Input: {src_a}\nOption A: {tgt_a}\nOption B: {tgt_b}\n"
            f"Answer with 'A' or 'B'.",
            "A", "H-PTB", "Cond_C"))

        payload.append(format_for_kaggle(
            f"D_{contrastive_idx}",
            f"One output follows the structural rule; one follows a surface pattern.\n"
            f"Input: {src_a}\nRule-based output: {tgt_a}\nSurface-based output: {tgt_b}\n"
            f"Which output relies on the compositional rule? Answer 'Rule-based' or 'Surface-based'.",
            "Rule-based", "H-PTB", "Cond_D_Alpha"))

        contrastive_idx += 1
        if contrastive_idx >= 200:
            break

    if contrastive_idx < 200:
        raise RuntimeError(
            f"H-PTB: only {contrastive_idx} contrastive pairs found; "
            f"need 200. Expand the train_data slice or check the dataset.")

    write_jsonl("hptb_N200.jsonl", payload)

# ──────────────────────────────────────────────────────────────────────────────
# TRACK 2 — METACOGNITION (H-MCB)  N=200
# Stage 1 (Prediction, k=5 types) + Stage 2 Performance (OOD N=200) + ID Control (N=200)
# ──────────────────────────────────────────────────────────────────────────────

def generate_hmcb() -> None:
    print("\n[2/5] Generating H-MCB (Metacognition) — N=200 ...")
    ood     = load_local_cogs("gen")[:200]     # raises if absent
    id_ctrl = load_local_cogs("train")[:200]   # raises if absent

    payload = []
    item_type_descriptions = [
        "applying a noun-phrase rule to novel agent-theme combinations",
        "applying a verb-phrase rule to unseen verb inflections",
        "applying a recursive embedding rule to three-clause structures",
        "applying a passivisation rule to held-out verbs",
        "applying a prepositional-phrase attachment rule to novel nouns",
    ]
    # Stage 1 — Prediction (T=0.7, k=5 meta-prompts)
    for i, desc in enumerate(item_type_descriptions):
        payload.append(format_for_kaggle(
            f"pred_{i}",
            f"You will complete 200 compositional semantic-parsing tasks involving {desc}.\n"
            f"These items were not present verbatim in your training data.\n"
            f"Estimate: what percentage (0-100) of these 200 tasks do you expect to answer correctly?\n"
            f"Respond with a single integer.",
            "[0-100]", "H-MCB", "Stage1_Prediction", temperature=0.7))

    # Stage 2 — OOD Performance (N=200)
    for i, item in enumerate(ood):
        payload.append(format_for_kaggle(
            f"perf_{i}",
            f"Map the following sentence to its logical form:\n{item['source']}",
            item["target"], "H-MCB", "Stage2_Performance"))

    # Stage 2 — In-Distribution Control (N=200)
    for i, item in enumerate(id_ctrl):
        payload.append(format_for_kaggle(
            f"id_ctrl_{i}",
            f"Map the following sentence to its logical form:\n{item['source']}",
            item["target"], "H-MCB", "Stage2_ID_Control"))

    write_jsonl("hmcb_N200.jsonl", payload)

# ──────────────────────────────────────────────────────────────────────────────
# TRACK 3 — ATTENTION (H-AFB)  N=200 per condition
# Three conditions: ID, OOD-Structural, OOD-Surface-Conflict
# ──────────────────────────────────────────────────────────────────────────────

def generate_hafb() -> None:
    print("\n[3/5] Generating H-AFB (Attention) — N=200 per condition ...")
    train_items = load_local_scan("train")     # raises if absent
    test_items  = load_local_scan("test")      # raises if absent

    test_copy = list(test_items)
    random.Random(42).shuffle(test_copy)
    test       = test_copy[:200]
    id_sample  = train_items[:200]

    if len(test) < 200:
        raise RuntimeError(f"H-AFB: SCAN test split has only {len(test)} items; need 200.")
    if len(id_sample) < 200:
        raise RuntimeError(f"H-AFB: SCAN train split has only {len(id_sample)} items; need 200.")

    payload = []

    # Condition 1 — In-Distribution (N=200)
    for i, item in enumerate(id_sample):
        payload.append(format_for_kaggle(
            f"id_{i}",
            f"Execute the following command sequence:\n{item['commands']}",
            item["actions"], "H-AFB", "ID"))

    # Condition 2 — OOD-Structural (N=200)
    for i, item in enumerate(test):
        payload.append(format_for_kaggle(
            f"struct_{i}",
            f"Execute the following command sequence:\n{item['commands']}",
            item["actions"], "H-AFB", "OOD_Structural"))

    # Condition 3 — OOD-Surface-Conflict (up to N=200; filtered on action content)
    conflict_items = [item for item in test if "JUMP JUMP" not in item["actions"]][:200]
    if len(conflict_items) < 200:
        raise RuntimeError(
            f"H-AFB: only {len(conflict_items)} conflict-eligible items found; need 200. "
            f"Expand test split or relax the filter.")
    for i, item in enumerate(conflict_items):
        payload.append(format_for_kaggle(
            f"conflict_{i}",
            f"Execute the following command sequence:\nRED {item['commands']}",
            item["actions"], "H-AFB", "OOD_SurfaceConflict"))

    write_jsonl("hafb_N200.jsonl", payload)

# ──────────────────────────────────────────────────────────────────────────────
# TRACK 4 — EXECUTIVE FUNCTIONS (H-DCB)  N=200
# Retrieval density: rho in {0.0, 0.2, 0.4, 0.6, 0.8, 1.0}  × 200 items
# + Inhibitory Conflict Task (BCR) N=200
# ──────────────────────────────────────────────────────────────────────────────

def generate_hdcb() -> None:
    print("\n[4/5] Generating H-DCB (Executive Functions) — N=200 ...")
    ood   = load_local_cogs("gen")[:200]       # raises if absent
    train = load_local_cogs("train")           # raises if absent

    rng        = random.Random(77)
    train_pool = [f"{t['source']} -> {t['target']}" for t in train[:200]]
    payload    = []
    rho_levels = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]

    # RAG density sweep: 200 items × 6 rho levels = 1 200 items
    for i, item in enumerate(ood):
        for rho in rho_levels:
            rho_label  = str(rho).replace(".", "p")
            n_examples = int(rho * 5)
            if n_examples == 0:
                prompt = f"Map to logical form: {item['source']}"
            else:
                context = " | ".join(rng.sample(train_pool, min(n_examples, len(train_pool))))
                prompt  = f"Context examples: {context}\n\nNow map to logical form: {item['source']}"
            payload.append(format_for_kaggle(
                f"rho{rho_label}_{i}",
                prompt,
                item["target"], "H-DCB", f"RAG_Rho{rho_label}"))

    # Inhibitory Conflict Task (BCR) — N=200
    bcr_count = 0
    for i, item in enumerate(ood):
        candidates = [x for x in ood if x["target"] != item["target"]]
        if not candidates:
            continue
        wrong_item = rng.choice(candidates)
        payload.append(format_for_kaggle(
            f"bcr_{i}",
            f"Hint (may be misleading): {wrong_item['target']}\n\n"
            f"Map to logical form: {item['source']}",
            item["target"], "H-DCB", "BCR_Inhibitory", temperature=0.0))
        bcr_count += 1

    if bcr_count < 200:
        raise RuntimeError(
            f"H-DCB BCR: only {bcr_count} conflict pairs generated; need 200. "
            f"Check OOD split for target diversity.")

    write_jsonl("hdcb_N200.jsonl", payload)

# ──────────────────────────────────────────────────────────────────────────────
# TRACK 5 — SOCIAL COGNITION (H-STB)  N=200
# Conditions: Schema, Fact, Mixed, ToM (per item) + Cooperative (N=200)
# ──────────────────────────────────────────────────────────────────────────────

def generate_hstb() -> None:
    print("\n[5/5] Generating H-STB (Social Cognition) — N=200 ...")
    ood = load_local_cogs("gen")[:200]         # raises if absent

    facts_str = (
        "'The cat ate the cake.' -> "
        "'eat.agent(x), cat(x) AND eat.theme(x), cake_x(x)' | "
        "'Liam read the book.' -> "
        "'read.agent(x), liam(x) AND read.theme(x), book_x(x)'"
    )
    schema_rule       = ("Verbs become event predicates; noun phrases become entity variables "
                         "linked by thematic roles (agent, theme, recipient).")
    mixed_schema_rule = "Verbs become event predicates; nouns become entity variables."

    payload = []
    for i, item in enumerate(ood):
        # Condition 1 — Schema-Transmission
        payload.append(format_for_kaggle(
            f"schema_{i}",
            f"Agent A has explained this rule: '{schema_rule}'\n"
            f"Using this structural rule, map the following sentence to its logical form:\n{item['source']}",
            item["target"], "H-STB", "Cond_Schema"))

        # Condition 2 — Fact-Transmission
        payload.append(format_for_kaggle(
            f"fact_{i}",
            f"Agent A has listed these observed mappings: {facts_str}.\n"
            f"Using these examples, map the following sentence to its logical form:\n{item['source']}",
            item["target"], "H-STB", "Cond_Fact"))

        # Condition 3 — Mixed
        payload.append(format_for_kaggle(
            f"mixed_{i}",
            f"Agent A has provided both a structural rule ('{mixed_schema_rule}') "
            f"and observed examples ({facts_str}).\n"
            f"Using both, map the following sentence to its logical form:\n{item['source']}",
            item["target"], "H-STB", "Cond_Mixed"))

        # ToM elicitation — predict Agent B's accuracy
        payload.append(format_for_kaggle(
            f"tom_{i}",
            f"Agent B has only seen the structural rule: '{mixed_schema_rule}'\n"
            f"Predict Agent B's accuracy (0-100%) on this item:\n{item['source']}",
            "[0-100]", "H-STB", "Cond_ToM", temperature=0.7))

    # Cooperative component — N=200 (all OOD items)
    for i, item in enumerate(ood):
        payload.append(format_for_kaggle(
            f"coop_{i}",
            f"Agent A knows verb-quantifier rules. Agent B knows preposition-noun rules.\n"
            f"This task requires both. Agent A communicates: "
            f"'Verbs map to event predicates with repetition per quantifier.'\n"
            f"Agent B communicates: 'Prepositions attach to noun variables.'\n"
            f"Jointly map to logical form:\n{item['source']}",
            item["target"], "H-STB", "Cond_Cooperative"))

    write_jsonl("hstb_N200.jsonl", payload)


if __name__ == "__main__":
    generate_hptb()
    generate_hmcb()
    generate_hafb()
    generate_hdcb()
    generate_hstb()
    print("\nAll N=200 payloads written to /kaggle/working/kaggle_payloads/")



[1/5] Generating H-PTB (Learning) ...
  Using real PCFG-SET data.
  Wrote 300 items -> /kaggle/working/kaggle_payloads/hptb_pilot.jsonl

[2/5] Generating H-MCB (Metacognition) ...
  Using real COGS data.
  Wrote 405 items -> /kaggle/working/kaggle_payloads/hmcb_pilot.jsonl

[3/5] Generating H-AFB (Attention) ...
  Using real SCAN data.
  Wrote 600 items -> /kaggle/working/kaggle_payloads/hafb_pilot.jsonl

[4/5] Generating H-DCB (Executive Functions) ...
  Using real COGS data.
  Wrote 1210 items -> /kaggle/working/kaggle_payloads/hdcb_pilot.jsonl

[5/5] Generating H-STB (Social Cognition) ...
  Using real COGS data.
  Wrote 810 items -> /kaggle/working/kaggle_payloads/hstb_pilot.jsonl

All payloads written to /kaggle/working/kaggle_payloads/


In [6]:
!pip install "protobuf>=5.29.3,<6.0.0"

  Using cached protobuf-5.29.6-cp38-abi3-manylinux2014_x86_64.whl.metadata (592 bytes)
Using cached protobuf-5.29.6-cp38-abi3-manylinux2014_x86_64.whl (320 kB)
  Attempting uninstall: protobuf
    Found existing installation: protobuf 7.34.1
    Uninstalling protobuf-7.34.1:
      Successfully uninstalled protobuf-7.34.1


In [8]:
# CELL 3: SDK EXECUTION — HARD-FAIL, NO SIMULATED FALLBACK
import os
import shutil
import subprocess
import sys

RESULTS_DIR = "/kaggle/working/kaggle_results" if _IS_KAGGLE else "./kaggle_results"
os.makedirs(RESULTS_DIR, exist_ok=True)

def get_sdk_path() -> str:
    """Resolve the kaggle-benchmarks binary. Raises if not found."""
    candidates = [
        shutil.which("kaggle-benchmarks"),
        "/opt/conda/bin/kaggle-benchmarks",
        os.path.expanduser("~/.local/bin/kaggle-benchmarks"),
        # Also check the active Python env's Scripts/bin
        os.path.join(os.path.dirname(sys.executable), "kaggle-benchmarks"),
    ]
    for path in candidates:
        if path and os.path.isfile(path):
            return path
    raise RuntimeError(
        "kaggle-benchmarks binary not found. "
        "Ensure 'pip install kaggle-benchmarks' completed successfully "
        "and the binary is on PATH (checked: " + ", ".join(str(c) for c in candidates) + ")."
    )

KAG_EXE = get_sdk_path()
print(f"Resolved SDK binary: {KAG_EXE}")

def run_track(payload_name: str, output_name: str) -> None:
    payload_path = f"/kaggle/working/kaggle_payloads/{payload_name}"
    output_path  = f"{RESULTS_DIR}/{output_name}"

    if not os.path.exists(payload_path):
        raise FileNotFoundError(
            f"Payload not found: {payload_path}. Run Cell 2 first.")

    command = [
        KAG_EXE, "run",
        "--benchmark", payload_path,
        "--models",    "gpt-4,claude-3-opus,gemini-1.5-pro",
        "--runs",      "5",
        "--output",    output_path,
    ]

    print(f"\n>>> INITIATING: {output_name}")
    process = subprocess.Popen(
        command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        print(line, end="")
    process.wait()

    if process.returncode != 0:
        raise RuntimeError(
            f"SDK exited with code {process.returncode} for {output_name}.")
    print(f"DONE: {output_name} saved to {output_path}")

# N=200 tracks
tracks = [
    ("hptb_N200.jsonl", "hptb_results.json"),
    ("hmcb_N200.jsonl", "hmcb_results.json"),
    ("hafb_N200.jsonl", "hafb_results.json"),
    ("hdcb_N200.jsonl", "hdcb_results.json"),
    ("hstb_N200.jsonl", "hstb_results.json"),
]

for p, o in tracks:
    run_track(p, o)

print("\n" + "="*50)
print("SDK PROCESSING COMPLETE — N=200 RESULTS WRITTEN.")
print("="*50)


Detected SDK Executable at: kaggle-benchmarks

>>> INITIATING: hptb_results.json


FileNotFoundError: [Errno 2] No such file or directory: 'kaggle-benchmarks'

In [ ]:
# CELL 4: METRICS CALCULATION — REAL RESULTS ONLY, NO SIMULATED FALLBACK

FD_SCORES = {"H-PTB": 0.750, "H-MCB": 0.800, "H-AFB": 0.830, "H-DCB": 0.830, "H-STB": 0.830}
DG_SCORES = {"H-PTB": 0.426, "H-MCB": 0.506, "H-AFB": 0.442, "H-DCB": 0.430, "H-STB": 0.509}

# H-Bar proxies for CI calculation (target: σ_A; confound: δ_A)
SIGMA_PROXIES = [0.65, 0.60, 0.50]
DELTA_PROXIES = [0.95, 0.96, 0.90]

def compute_reliability(model_runs: list[float]) -> float:
    """RA = 1 / (1 + CV²). Bounded [0,1]. Edge case: mean=0 → 0."""
    scores = np.array(model_runs)
    if np.mean(scores) == 0:
        return 0.0
    cv_squared = np.var(scores, ddof=0) / (np.mean(scores) ** 2)
    return 1.0 / (1.0 + cv_squared)

def compute_construct_isolation(
    target_scores: list[float],
    target_proxy_vals: list[float],
    confound_vals: list[float],
) -> float:
    """CI = |Corr(score, target)| / |Corr(score, confound)|, clamped [0,1]."""
    corr_target,   _ = pearsonr(target_scores, target_proxy_vals)
    corr_confound, _ = pearsonr(target_scores, confound_vals)
    if corr_confound == 0 or np.isnan(corr_confound):
        return 1.0
    return min(1.0, abs(corr_target) / abs(corr_confound))

def analyze_track_results(track_name: str, results_file: str) -> None:
    print(f"\n{'='*30}\n{track_name} VALIDITY REPORT\n{'='*30}")

    with open(results_file, "r") as f:        # raises FileNotFoundError if absent
        data = json.load(f)

    # Hard-key access — raises KeyError if SDK output is malformed
    claude_runs = data["claude-3-opus"]
    gpt4_runs   = data["gpt-4"]
    gemini_runs = data["gemini-1.5-pro"]

    # Validate run counts
    for model_name, runs in [
        ("claude-3-opus", claude_runs),
        ("gpt-4",         gpt4_runs),
        ("gemini-1.5-pro", gemini_runs),
    ]:
        if len(runs) != 5:
            raise ValueError(
                f"{track_name}: expected 5 runs for {model_name}, got {len(runs)}.")

    ra_mean = np.mean([
        compute_reliability(claude_runs),
        compute_reliability(gpt4_runs),
        compute_reliability(gemini_runs),
    ])

    model_avgs = [np.mean(claude_runs), np.mean(gpt4_runs), np.mean(gemini_runs)]
    ci_score   = compute_construct_isolation(model_avgs, SIGMA_PROXIES, DELTA_PROXIES)

    fd = FD_SCORES[track_name]
    dg = DG_SCORES[track_name]
    va_total = ci_score * fd * dg * ra_mean

    print(f"  CI (Construct Isolation): {ci_score:.3f}   [Target > 0.60]")
    print(f"  FD (Format Diversity)   : {fd:.3f}   [Target > 0.55]")
    print(f"  DG (Difficulty Gradient): {dg:.3f}   [Target > 0.40]")
    print(f"  RA (Reliability)        : {ra_mean:.3f}   [Target > 0.75]")
    print("  -------------------------------------")
    print(f"  V_A(B,f,t) COMPOSITE    : {va_total:.3f}   [Target > 0.20]")

    status = "PASS" if va_total > 0.20 else "FAIL"
    print(f"  >>> STATUS: {status}.")
    if status == "FAIL":
        raise RuntimeError(
            f"{track_name} V_A={va_total:.3f} below threshold 0.20. Redesign required.")

RESULTS_DIR = "/kaggle/working/kaggle_results"
analyze_track_results("H-PTB", f"{RESULTS_DIR}/hptb_results.json")
analyze_track_results("H-MCB", f"{RESULTS_DIR}/hmcb_results.json")
analyze_track_results("H-AFB", f"{RESULTS_DIR}/hafb_results.json")
analyze_track_results("H-DCB", f"{RESULTS_DIR}/hdcb_results.json")
analyze_track_results("H-STB", f"{RESULTS_DIR}/hstb_results.json")
